# 05 - Survival analysis: when do players die, and what shifts the risk?

**Time** = `last_round` (the round a player was eliminated). **Event** = eliminated. The winner is *censored*: they never died, the game ended.

**What this data can answer:** when elimination risk peaks, what finishing placement you can expect once you have reached a given round, and whether comp or contestation changes the elimination hazard.

**What it cannot answer:** whether slamming items or rolling down early saves HP versus holding gold for interest. The Riot match data has no round-by-round HP, gold or items, only each player's final board. The decision engine will need explicit game rules or assumptions for HP.

Install once for the Cox model: `python -m pip install lifelines`. Everything before section 4 runs without it.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = next(p for p in [Path("data"), Path("../data"), Path("D:/VS Code/data")] if (p / "boards_clustered.csv").exists())
boards = pd.read_csv(DATA / "boards_clustered.csv", index_col="board_id")
boards["event"] = (boards["placement"] > 1).astype(int)   # winner is censored, everyone else was eliminated

ARCH = "archetype" if "archetype" in boards.columns else "cluster"
FLEX = [-1, 10]

# contestation, same definition as notebook 03
family = boards["cluster"].replace({13: 12, 4: 5}).rename("family")
valid = ~boards["cluster"].isin(FLEX)
boards["others_same_comp"] = np.nan
boards.loc[valid, "others_same_comp"] = (
    boards[valid].groupby(["match_id", family[valid]])["placement"].transform("size") - 1)

print("Boards:", len(boards), "| matches:", boards["match_id"].nunique())
print("Eliminated:", int(boards["event"].sum()), "| censored (winners):", int((boards["event"] == 0).sum()))
print()
print(boards["last_round"].describe().round(1))
print()
same = boards[boards["placement"].isin([1, 2])].groupby("match_id")["last_round"].nunique()
print("Share of matches where 1st and 2nd have the same last_round:", round(float((same == 1).mean()), 3))
print()
print("Share of exactly-zero values (all zeros means the column is unusable):")
print((boards[["players_eliminated", "total_damage_to_players"]] == 0).mean().round(3))

## 1. When does each placement die?
Placement is just the order of elimination, so `last_round` and placement are mechanically linked. This table shows how many rounds each finishing position typically survives.

In [ ]:
by_place = boards.groupby("placement")["last_round"].agg(["count", "min", "median", "max"])
by_place

## 2. Kaplan-Meier survival and the hazard by round
- `survival` = share of players still alive after that round.
- `hazard` = of the players alive at the start of that round, the share who die in it. The peaks are the dangerous rounds.

Both are computed by hand, no extra package needed.

In [ ]:
def km_table(df, time="last_round", event="event"):
    t = df.groupby(time).agg(n_events=(event, "sum"), n_total=(event, "size")).sort_index()
    t["at_risk"] = t["n_total"][::-1].cumsum()[::-1]
    t["hazard"] = t["n_events"] / t["at_risk"]
    t["survival"] = (1 - t["hazard"]).cumprod()
    return t

def km_median(t):
    below = t.index[t["survival"] <= 0.5]
    return float(below[0]) if len(below) else np.nan

overall = km_table(boards)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
x = [overall.index.min() - 1] + list(overall.index)
y = [1.0] + list(overall["survival"])
ax[0].step(x, y, where="post")
ax[0].set_xlabel("round (last_round)")
ax[0].set_ylabel("share of players still alive")
ax[0].set_title("Kaplan-Meier survival, all boards")
ax[1].bar(overall.index, overall["hazard"], color="indianred")
ax[1].set_xlabel("round (last_round)")
ax[1].set_ylabel("hazard (death rate among those alive)")
ax[1].set_title("Elimination hazard by round")
plt.tight_layout()
plt.show()
print("Median survival round:", km_median(overall))
overall.round(3)

## 3. Once you reach round R, what should you expect?
For each round: how many players reached it, their average final placement, and the share who finished top 4.
This is a descriptive table, not a causal one, but it prices what surviving one more round is worth.

In [ ]:
rows = []
for r in sorted(boards["last_round"].unique()):
    alive = boards[boards["last_round"] >= r]
    rows.append(dict(round=r, reached=len(alive),
                     mean_placement=round(alive["placement"].mean(), 2),
                     top4_rate=round((alive["placement"] <= 4).mean(), 3)))
alive_at_round = pd.DataFrame(rows).set_index("round")
alive_at_round

## 4. Does comp or contestation change the hazard?
Flex and catch-all boards are excluded, because they are not a single comp.

**Read this carefully.** The archetype is read from the *final* board, so a player who dies early never finished their comp. Archetype effects here are descriptive. Board strength (items, stars, level) is deliberately not used as a covariate: it grows over time, so it would mostly measure how long the player survived.

In [ ]:
non_flex = boards[~boards["cluster"].isin(FLEX)].copy()
top = non_flex[ARCH].value_counts().head(6).index

fig, ax = plt.subplots(figsize=(9, 6))
rows = []
for a in top:
    d = non_flex[non_flex[ARCH] == a]
    t = km_table(d)
    ax.step([t.index.min() - 1] + list(t.index), [1.0] + list(t["survival"]), where="post", label=f"{a} (n={len(d)})")
    rows.append(dict(archetype=a, boards=len(d), mean_placement=round(d["placement"].mean(), 2),
                     median_last_round=d["last_round"].median(), km_median_round=km_median(t)))
ax.set_xlabel("round (last_round)")
ax.set_ylabel("share still alive")
ax.set_title("Survival by archetype (six largest)")
ax.legend(fontsize=8)
plt.show()
pd.DataFrame(rows).sort_values("mean_placement")

In [ ]:
try:
    from lifelines import CoxPHFitter
    from lifelines.statistics import multivariate_logrank_test, proportional_hazard_test
    HAVE_LL = True
except ImportError:
    HAVE_LL = False
    print("lifelines is not installed. Run:  python -m pip install lifelines   then restart the kernel.")

if HAVE_LL:
    lr = multivariate_logrank_test(non_flex["last_round"], non_flex[ARCH], non_flex["event"])
    print(f"Log-rank test across all archetypes: chi2 = {lr.test_statistic:.1f}, p = {lr.p_value:.4g}")

In [ ]:
if HAVE_LL:
    ref = non_flex[ARCH].value_counts().index[0]
    dummies = pd.get_dummies(non_flex[ARCH], prefix="a").astype(float).drop(columns=f"a_{ref}")
    cox_df = pd.concat([non_flex[["last_round", "event", "match_id", "others_same_comp"]], dummies], axis=1)

    cph = CoxPHFitter()
    cph.fit(cox_df, duration_col="last_round", event_col="event", cluster_col="match_id")  # robust SE by match
    print("Reference archetype:", ref)
    print("Concordance:", round(cph.concordance_index_, 3))
    out = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3)
    out.columns = ["hazard_ratio", "ci_low", "ci_high", "p"]
    display(out.sort_values("hazard_ratio"))

**Reading the hazard ratios:** below 1 means a lower risk of elimination at any given round than the reference archetype, above 1 means higher. `others_same_comp` is the effect of each extra player on the same comp.
Standard errors are clustered by match, because the eight players in a game are not independent.

In [ ]:
if HAVE_LL:
    try:
        ph = proportional_hazard_test(cph, cox_df, time_transform="rank")
        display(ph.summary.round(3))
    except Exception as e:
        print("Proportional-hazards test could not run:", type(e).__name__, e)

## 5. Exploratory: dying with unspent gold
Closest thing in this data to the "building interest while HP drains" question. Among players eliminated early (the earliest quarter of deaths), what share still held a lot of gold?
Gold left is measured at elimination, so this shows how often early deaths happened with the economy unspent. It does not show *why*.

In [ ]:
died = boards[boards["event"] == 1]
cut = died["last_round"].quantile(0.25)
early = died[died["last_round"] <= cut]
rows = []
for th in (20, 30, 50):
    rows.append({"gold_left": f">= {th}", "early_deaths": f"{(early['gold_left'] >= th).mean():.1%}",
                 "all_deaths": f"{(died['gold_left'] >= th).mean():.1%}"})
print(f"Early deaths = last_round <= {cut:.0f}  (n = {len(early)})")
pd.DataFrame(rows)

In [ ]:
overall.to_csv(DATA / "hazard_by_round.csv")
alive_at_round.to_csv(DATA / "alive_at_round.csv")
print("saved hazard_by_round.csv and alive_at_round.csv")

In [ ]:
rng = np.random.default_rng(0)
mid = boards["match_id"].values
ids = np.unique(mid)
idx = {m: np.where(mid == m)[0] for m in ids}
gaps = []
for _ in range(500):
    pick = rng.choice(ids, len(ids), replace=True)
    d = boards.iloc[np.concatenate([idx[m] for m in pick])]
    gaps.append(d.loc[d.cluster == 13, "placement"].mean() - d.loc[d.cluster == 12, "placement"].mean())
print(np.percentile(gaps, [2.5, 50, 97.5]).round(2))

## What to take from this
- The hazard curve tells you which rounds kill the lobby. The decision engine can use it as baseline danger per round.
- `alive_at_round` is the value of survival: expected placement and top-4 rate once you have reached each round.
- Hazard ratios show whether comp or contestation shifts risk. Expect them to be modest if archetype barely predicts placement, as in notebook 03.
- Missing for the engine: HP. It has to come from game rules (damage per lost fight by stage) or user input, not from this data.